# Do Qwen's explanations reflect what changes its answers?

## Prepare e-SNLI

Freeze 1,000 test examples and select 10 development examples for the later smoke test. This section runs on CPU. On Kaggle, enable internet for the first run. On your computer, use a Python 3.10–3.13 Jupyter kernel.

In [ ]:
%pip install -q spacy==3.8.7 vaderSentiment==3.3.2 https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

### Settings

The seed, dataset revision, sample sizes, and output path are fixed here.

In [ ]:
import csv
import json
import math
import random
import re
import sys
from collections import Counter
from hashlib import sha256
from importlib.metadata import version
from pathlib import Path
from shutil import copyfileobj
from urllib.request import urlopen

SEED = 42
SAMPLE_SIZE = 1000
SMOKE_SIZE = 10
OUTPUT = Path("data/prepared")
DATASET_REVISION = "7b585a3f077fdea899780eb0473940522ae44a2e"
SAMPLE_HASHES = {
    "frozen_examples.csv": "02ec2eccb7a93f5dd8682b7c2bfbd339b8aefbea79d7bd45ff736ced4195b2d2",
    "smoke_examples.csv": "0fac2eeeed098d7bec38f52346da45f0ff77d5f2ec1277aa224f7d648cef410e",
    "edit_proposals.csv": "b4e026a485d729265c03f22195b9c931b3ac2a745217b345570689cc7ebc6969",
    "smoke_edits.csv": "f6a1eb8102aae7bcb61a312d59c107b671af2d3c918dd5f04ac8bf3992afcd68",
}
REPOSITORY = "OanaMariaCamburu/e-SNLI"
SPLITS = {"train": ("train_1", "train_2"), "test": ("test",), "validation": ("dev",)}
LABELS = ("entailment", "neutral", "contradiction")
ROLES = ("premise", "hypothesis")
FAMILIES = ("insertion", "deletion")
WORD = re.compile(r"[^\W\d_]+(?:['’][^\W\d_]+)*")
MANIFEST_PATH = OUTPUT / "manifest.json"

### 1. Sample the data and count training words

We use the official [e-SNLI CSVs](https://github.com/OanaMariaCamburu/e-SNLI), keeping their original pair IDs and splits. We deduplicate evaluation pairs by their text after normalizing case and whitespace. The same seed and dataset revision give the same sample. The revision is pinned to the first Kaggle run. The exported test and development CSVs must match that run's SHA-256 hashes.

For IDF, one training pair is one document. We count each lowercase word once per pair, keeping apostrophes within words. Unseen words have a document frequency of zero. Target rarity is the mean IDF of its words.

In [ ]:
def word_tokens(text):
    return WORD.findall(text.casefold())

def split_rows(split, revision):
    cache = OUTPUT.parent / ".cache" / revision
    cache.mkdir(parents=True, exist_ok=True)
    seen = set()
    for part in SPLITS[split]:
        filename = f"esnli_{part}.csv"
        path = cache / filename
        if not path.exists():
            url = f"https://raw.githubusercontent.com/{REPOSITORY}/{revision}/dataset/{filename}"
            temporary = path.with_suffix(".tmp")
            with urlopen(url, timeout=120) as response, temporary.open("wb") as stream:
                copyfileobj(response, stream)
            temporary.replace(path)
        with path.open(encoding="utf-8-sig", newline="") as stream:
            reader = csv.DictReader(stream)
            headers = {name.strip().casefold(): name for name in (reader.fieldnames or [])}
            required = {"pairid", "gold_label", "sentence1", "sentence2"}
            if not required <= headers.keys():
                raise ValueError(f"Unexpected e-SNLI columns in {filename}: {list(headers)}")
            for row in reader:
                source_id, label, premise, hypothesis = (
                    (row[headers[key]] or "").strip()
                    for key in ("pairid", "gold_label", "sentence1", "sentence2")
                )
                if source_id and source_id not in seen and premise and hypothesis and label in LABELS:
                    seen.add(source_id)
                    yield dict(source_id=source_id, split=split, premise=premise,
                               hypothesis=hypothesis, gold_label=label)

def pair_key(row):
    return tuple(" ".join(row[role].casefold().split()) for role in ROLES)

def sample_split(split, size, revision, excluded=()):
    seen = {pair_key(row) for row in excluded}
    unique = []
    for row in sorted(split_rows(split, revision), key=lambda item: item["source_id"]):
        key = pair_key(row)
        if key not in seen:
            seen.add(key)
            unique.append(row)
    return random.Random(f"{SEED}:{split}").sample(unique, size)

def training_word_counts(revision):
    counts = Counter()
    size = 0
    for row in split_rows("train", revision):
        counts.update(set(word_tokens(row["premise"] + "\n" + row["hypothesis"])))
        size += 1
    if not size:
        raise ValueError("The training split is empty.")
    return counts, size

revision = DATASET_REVISION
test_examples = sample_split("test", SAMPLE_SIZE, revision)
smoke_examples = sample_split("validation", SMOKE_SIZE, revision, excluded=test_examples)
document_frequency, training_size = training_word_counts(revision)

### 2. Define the edit rules

Add an adjective, an adverb, or a city when it fits. If those rules find nothing, try adding a time expression.
For deletion, remove a simple modifier or a phrase at the end of the sentence.

In [ ]:
import spacy

NLP = spacy.load("en_core_web_sm")
PEOPLE = set(
    "man woman boy girl child kid person player worker rider skier runner cyclist "
    "male female adult teen teenager student athlete dancer singer musician performer "
    "pedestrian tourist visitor farmer driver painter hiker cook swimmer snowboarder".split()
)
OBJECTS = set(
    "shirt jacket coat hat dress sweater pants bicycle bike car truck bag ball umbrella "
    "cap helmet scarf vest glove sock shoe boot belt skirt shorts backpack purse "
    "bus van motorcycle scooter wagon skateboard snowboard boat bench chair flag sign kite cup".split()
)
ACTIONS = set(
    "walk run ride dance play work read write talk speak sing swim jump skate ski paint move "
    "climb hike skate stand sit lie wait rest jog race march perform practice exercise "
    "cook eat drink draw watch pose carry hold throw catch kick swing wave stretch".split()
)
MANNER = ("slowly", "quickly", "quietly", "eagerly", "carefully", "casually", "reluctantly")
OPTIONAL_ADVERBS = set(MANNER) | set(
    "happily sadly silently calmly loudly gently steadily together alone nearby "
    "outside outdoors indoors inside today yesterday".split()
)
CITIES = ("Paris", "Berlin", "Oslo", "Lisbon", "Nairobi", "Accra")
TIMES = ((" at noon", "noon"), (" at dusk", "dusk"), (" after sunrise", "sunrise"))
ADJUNCTS = {"in", "at", "on", "near", "under", "beside", "during", "behind", "across",
            "around", "along", "above", "beneath", "outside", "inside", "by"}

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Edit:
    start: int
    end: int
    replacement: str
    target: str
    method: str

def agrees_with_article(previous, following):
    if previous not in {"a", "an"}:
        return True
    return following.isalpha() and (previous == "an") == (following[0].lower() in "aeiou")

def insertions(doc):
    for token in doc:
        if not token.i or not doc.text[token.idx - 1].isspace():
            continue
        previous = doc[token.i - 1]
        has_modifier = any(child.dep_ in {"amod", "compound", "nummod"} for child in token.children)
        if token.pos_ == "NOUN" and previous.pos_ in {"DET", "PRON"} and not has_modifier:
            adjectives = ()
            if token.lemma_ in PEOPLE:
                adjectives = ("tall", "young", "cheerful", "sad", "anxious", "weary")
            elif token.lemma_ in OBJECTS:
                adjectives = ("red", "blue", "pink", "purple", "crimson", "turquoise")
            for adjective in adjectives:
                if agrees_with_article(previous.lower_, adjective):
                    yield Edit(token.idx, token.idx, adjective + " ", adjective, "adjective")
        if token.dep_ != "ROOT" or token.pos_ != "VERB" or token.lemma_ not in ACTIONS:
            continue
        if not any(child.dep_ in {"advmod", "prt"} for child in token.children):
            for adverb in MANNER:
                yield Edit(token.idx, token.idx, adverb + " ", adverb, "adverb")
        has_place = any(entity.label_ in {"GPE", "LOC", "FAC"} for entity in doc.ents)
        if not has_place and not any(child.dep_ == "prep" for child in token.children):
            tail = re.search(r"\s*\.?$", doc.text).start()
            if tail and doc.text[tail - 1].isalnum() and not re.search(r"[,;:\"()]", doc.text):
                for city in CITIES:
                    yield Edit(tail, tail, " in " + city, city, "city")

def time_insertions(doc):
    has_time = any(entity.label_ in {"DATE", "TIME"} for entity in doc.ents)
    if has_time or len(list(doc.sents)) != 1 or re.search(r"[,;:\"()]", doc.text):
        return
    roots = [token for token in doc if token.dep_ == "ROOT"]
    if not any(token.pos_ == "VERB" and token.lemma_ in ACTIONS for token in roots):
        return
    tail = re.search(r"\s*\.?$", doc.text).start()
    if tail and doc.text[tail - 1].isalnum():
        for phrase, target in TIMES:
            yield Edit(tail, tail, phrase, target, "time")

def deletions(doc):
    for token in doc:
        start, end = token.idx, token.idx + len(token)
        if not token.i or not doc.text[start - 1].isspace():
            continue
        simple_adjective = token.dep_ == "amod" and token.pos_ == "ADJ"
        simple_adverb = token.dep_ == "advmod" and token.lower_ in OPTIONAL_ADVERBS
        if (simple_adjective or simple_adverb) and not any(token.children):
            if simple_adverb and re.fullmatch(r"\s*[.!?]?", doc.text[end:]):
                yield Edit(start - 1, end, "", token.text, "modifier")
            elif token.i + 1 < len(doc):
                following = doc[token.i + 1]
                previous = doc[token.i - 1].lower_
                if end < following.idx and WORD.search(token.text) and agrees_with_article(previous, following.text):
                    yield Edit(start, following.idx, "", token.text, "modifier")
        if token.dep_ != "prep" or token.lower_ not in ADJUNCTS:
            continue
        if token.head.pos_ != "VERB" or token.head.lemma_ not in ACTIONS:
            continue
        phrase = list(token.subtree)
        if len(phrase) < 2 or phrase[0].i != token.i or phrase[-1].i - token.i + 1 != len(phrase):
            continue
        if any(part.dep_ in {"cc", "conj", "punct"} for part in phrase):
            continue
        phrase_end = phrase[-1].idx + len(phrase[-1])
        if re.fullmatch(r"\s*[.!?]?", doc.text[phrase_end:]):
            target = doc.text[start:phrase_end]
            if word_tokens(target):
                yield Edit(start - 1, phrase_end, "", target, "trailing_adjunct")

### 3. Measure the edited detail

We record its position in the input, its sentiment, its word rarity, and whether it overlaps a named entity.
The span uses the edited sentence for an insertion and the original sentence for a deletion.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

SENTIMENT = SentimentIntensityAnalyzer()
NAME_TYPES = {"PERSON", "NORP", "FAC", "ORG", "GPE", "LOC", "PRODUCT", "EVENT",
              "WORK_OF_ART", "LAW", "LANGUAGE"}
NEGATION = re.compile(r"\b(?:not|never|no)\b|n['’]t\b", re.IGNORECASE)


def target_features(doc, start, end, target, words_before, total_words):
    entities = sorted({entity.label_ for entity in doc.ents
                       if entity.label_ in NAME_TYPES and entity.start_char < end and entity.end_char > start})
    token_indexes = [i for i, word in enumerate(WORD.finditer(doc.text))
                     if word.start() < end and word.end() > start]
    midpoint = (token_indexes[0] + token_indexes[-1] + 1) / 2
    position = (words_before + midpoint) / total_words
    score = SENTIMENT.polarity_scores(target)["compound"]
    polarity = "neutral"
    if score <= -0.05:
        polarity = "negative"
    elif score >= 0.05:
        polarity = "positive"
    idfs = [math.log((training_size + 1) / (document_frequency[word] + 1)) + 1
            for word in word_tokens(target)]
    return dict(named_entity=int(bool(entities)), entity_types="|".join(entities),
                vader_score=score, polarity=polarity, negation_in_target=int(bool(NEGATION.search(target))),
                position=position, position_bin=("early", "middle", "late")[min(2, int(position * 3))],
                idf=sum(idfs) / len(idfs))

### 4. Prepare the proposals

Choose one candidate per edit family using the fixed seed. Record failed attempts alongside the proposals.

In [ ]:
def choose_edit(docs, family, edit_id, original_words):
    insertion = family == "insertion"
    rules = (insertions, time_insertions) if insertion else (deletions,)
    for rule in rules:
        candidates = {}
        for role, doc in docs.items():
            edits = list(rule(doc))
            if insertion:
                edits = [edit for edit in edits if original_words.isdisjoint(word_tokens(edit.target))]
            if edits:
                candidates[role] = edits
        if candidates:
            rng = random.Random(f"{SEED}:{edit_id}")
            role = rng.choice(list(candidates))
            edits = candidates[role]
            method = rng.choice(sorted({edit.method for edit in edits}))
            return role, rng.choice([edit for edit in edits if edit.method == method])
    return None

def make_proposal(row, docs, family):
    insertion = family == "insertion"
    proposal = {**row, "edit_id": f"{row['source_id']}:{family}", "family": family,
                "edited_premise": row["premise"], "edited_hypothesis": row["hypothesis"]}
    original_words = set(word_tokens(row["premise"] + "\n" + row["hypothesis"]))
    selection = choose_edit(docs, family, proposal["edit_id"], original_words)
    if selection is None:
        return {**proposal, "status": "rejected", "rejection_reason": "no eligible rule-based edit"}

    role, edit = selection
    original = row[role]
    edited = original[:edit.start] + edit.replacement + original[edit.end:]
    span_doc = NLP(edited) if insertion else docs[role]
    changed_text = edit.replacement if insertion else original[edit.start:edit.end]
    start = edit.start + changed_text.index(edit.target)
    end = start + len(edit.target)
    assert span_doc.text[start:end] == edit.target
    proposal[f"edited_{role}"] = edited
    reference = {key: proposal[f"edited_{key}"] if insertion else row[key] for key in ROLES}
    premise_words = len(word_tokens(reference["premise"]))
    total_words = premise_words + len(word_tokens(reference["hypothesis"]))
    features = target_features(span_doc, start, end, edit.target,
                               premise_words if role == "hypothesis" else 0, total_words)
    return {**proposal, **features, "status": "proposed", "method": edit.method, "role": role,
            "target": edit.target, "target_start": start, "target_end": end,
            "span_text": "edited" if insertion else "original", "patch_start": edit.start,
            "patch_end": edit.end, "replacement": edit.replacement}

def propose_edits(examples):
    texts = (row[role] for row in examples for role in ROLES)
    documents = iter(NLP.pipe(texts, batch_size=64))
    proposals = []
    for row in examples:
        docs = {role: next(documents) for role in ROLES}
        proposals.extend(make_proposal(row, docs, family) for family in FAMILIES)
    return proposals

test_edits = propose_edits(test_examples)
smoke_edits = propose_edits(smoke_examples)

### 5. Freeze the proposals and review 20 edits

We divide the test edit IDF scores into three groups within each edit family using the nearest-rank 33rd and 67th percentiles. Equal scores stay together. The same cutoffs are used for the development examples.

Review ten insertions and ten deletions in `data/prepared/edit_audit.csv`. Compare each original and edited pair. Record whether it reads naturally, makes sense, changes the meaning, might change the correct label, and has a correct entity annotation. Use `accept` or `reject` in `review_decision` and explain any rejection in `review_notes`. The audit deliberately includes up to three named-entity edits per family and attempts to cover every edit method. It checks edit quality and annotation errors and is separate from the model evaluation.

In [ ]:
def assign_rarity_and_select_audit():
    thresholds = {}
    audit = []
    for family in FAMILIES:
        proposed = [row for row in test_edits if row["family"] == family and row["status"] == "proposed"]
        scores = sorted(row["idf"] for row in proposed)
        if not scores:
            thresholds[family] = None
            continue
        low, high = (scores[math.ceil(len(scores) * q) - 1] for q in (1 / 3, 2 / 3))
        thresholds[family] = [low, high]
        for row in test_edits + smoke_edits:
            if row["family"] == family and row["status"] == "proposed":
                row["rarity_bin"] = "common" if row["idf"] <= low else "middle" if row["idf"] <= high else "rare"

        rng = random.Random(f"{SEED}:audit:{family}")
        entities = [row for row in proposed if row["named_entity"]]
        picked = rng.sample(entities, min(3, len(entities)))
        missing_methods = sorted({row["method"] for row in proposed} - {row["method"] for row in picked})
        for method in missing_methods:
            picked.append(rng.choice([row for row in proposed if row["method"] == method]))
        picked_ids = {row["edit_id"] for row in picked}
        other = [row for row in proposed if row["edit_id"] not in picked_ids]
        picked.extend(rng.sample(other, min(10 - len(picked), len(other))))
        audit_columns = ("edit_id", "family", "method", "role", "target", "premise", "hypothesis",
                         "edited_premise", "edited_hypothesis", "gold_label", "entity_types")
        audit.extend({**{key: row[key] for key in audit_columns},
                      "grammatical": "", "coherent": "", "meaning_changed": "",
                      "gold_label_may_change": "", "entity_annotation_correct": "",
                      "review_decision": "", "review_notes": ""} for row in picked)
    return thresholds, audit

thresholds, audit = assign_rarity_and_select_audit()

### 6. Save the outputs

The CSVs store the examples, edits, and audit. The manifest records coverage and verifies the frozen sample.

In [ ]:
def export_csv(path, rows, expected_hash=None):
    fields = list(dict.fromkeys(field for row in rows for field in row))
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fields, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    digest = sha256(temporary.read_bytes()).hexdigest()
    if expected_hash is not None and digest != expected_hash:
        temporary.unlink()
        raise ValueError(f"Regenerated data differs from the frozen version of {path.name}.")
    temporary.replace(path)
    return digest

files = {"frozen_examples.csv": test_examples, "edit_proposals.csv": test_edits,
         "smoke_examples.csv": smoke_examples, "smoke_edits.csv": smoke_edits}
OUTPUT.mkdir(parents=True, exist_ok=True)
previous = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else None
if previous:
    previous_settings = (previous["seed"], previous["test_size"], previous["smoke_size"])
    if previous_settings != (SEED, SAMPLE_SIZE, SMOKE_SIZE):
        raise ValueError("The output directory already contains a sample frozen with different settings.")
expected_hashes = {**(previous["file_sha256"] if previous else {}), **SAMPLE_HASHES}
hashes = {name: export_csv(OUTPUT / name, rows, expected_hashes.get(name))
          for name, rows in files.items()}
audit_path = OUTPUT / "edit_audit.csv"
if not audit_path.exists() and audit:
    export_csv(audit_path, audit)

counts = Counter(row["gold_label"] for row in test_examples)
proposals = Counter(f"{row['family']}_{row['status']}" for row in test_edits)
edits_per_example = Counter(row["source_id"] for row in test_edits if row["status"] == "proposed")
coverage = Counter(edits_per_example[row["source_id"]] for row in test_examples)
manifest = {
    "dataset_source": f"https://github.com/{REPOSITORY}",
    "dataset_revision": revision, "seed": SEED,
    "test_split": "test", "smoke_split": "validation",
    "test_size": len(test_examples), "smoke_size": len(smoke_examples),
    "training_documents": training_size,
    "label_counts": dict(counts),
    "label_proportions": {label: counts[label] / len(test_examples) for label in LABELS},
    "proposal_counts": dict(proposals), "idf_thresholds": thresholds,
    "examples_by_proposal_count": {str(count): coverage[count] for count in range(3)},
    "versions": {"python": sys.version.split()[0], **{
        name: version(name) for name in ("spacy", "en-core-web-sm", "vaderSentiment")
    }},
    "file_sha256": hashes,
}
if previous is None:
    temporary = MANIFEST_PATH.with_suffix(".tmp")
    temporary.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    temporary.replace(MANIFEST_PATH)

print(f"Frozen test examples: {len(test_examples)}. Validation smoke examples: {len(smoke_examples)}.")
print(f"Output directory: {OUTPUT.resolve()}")
print("Test label counts:", dict(counts))
print("Edit proposal counts:", dict(proposals))
print("Examples with zero, one, or two proposals:", [coverage[count] for count in range(3)])
print(f"Audit rows available: {len(audit)}. Review at least 20 before running the main evaluation.")
if len(audit) < 20:
    print("The audit needs more eligible edits.")

## Check Qwen on the validation sample

Use a Kaggle session with **GPU T4 x2** and internet enabled. This check uses the ten frozen validation pairs and their proposed edits. Each pair runs in three answer orders. The output keeps the letter, its mapped label, and the response text so the mapping can be checked.

In [ ]:
%pip install -q transformers==4.57.3 accelerate==1.12.0

### Build the prompts and read the answers

Qwen explains its choice, then gives one letter on the last line. We read those two fields in order and use the option order in the prompt to turn the letter back into a label. Empty, malformed, and truncated answers are recorded as failures.

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"
MODEL_REVISION = "c4991a667c9137cff07024144d448d2afe76abc4"
BATCH_SIZE = 8
MAX_NEW_TOKENS = 96
SMOKE_OUTPUT = Path("results/smoke/greedy")

LETTERS = ("A", "B", "C")
OPTION_ORDERS = [LABELS[i:] + LABELS[:i] for i in range(len(LABELS))]
LABEL_DEFINITIONS = {
    "entailment": "The premise makes the hypothesis true.",
    "neutral": "The premise leaves the hypothesis undecided.",
    "contradiction": "The premise rules out the hypothesis.",
}
EXPLANATION_START = re.compile(r"\s*(?:\*\*)?Explanation:[ \t]*(?:\*\*)?[ \t]*", re.IGNORECASE)
LABEL_LINE = re.compile(
    r"[ \t]*(?:\*\*)?Label:[ \t]*(?:\*\*)?[ \t]*(?:\*\*)?([ABC])(?:\*\*)?\.?",
    re.IGNORECASE,
)
EXTRA_FIELD = re.compile(r"(?im)^[ \t]*(?:\*\*)?(?:Explanation|Label):|</?think>")


def make_prompt(premise, hypothesis, order):
    choices = "\n".join(f"{letter}. {label}: {LABEL_DEFINITIONS[label]}"
                        for letter, label in zip(LETTERS, order))
    return (
        f"Premise: {premise}\nHypothesis: {hypothesis}\n\n{choices}\n\n"
        "Use the premise to judge the hypothesis. Explain your decision in one or two sentences. "
        "Then give one choice letter on the last line. Use this format:\n"
        "Explanation: <explanation>\nLabel: <A, B, or C>"
    )

def parse_response(text, finished, order):
    empty_fields = dict(explanation="", label_letter="", predicted_label="")
    if not finished:
        return dict(status="truncated", **empty_fields)
    if not text.strip():
        return dict(status="empty", **empty_fields)

    explanation_block, _, label_line = text.rstrip().rpartition("\n")
    start = EXPLANATION_START.match(explanation_block)
    label = LABEL_LINE.fullmatch(label_line)
    if start is None or label is None:
        return dict(status="malformed", **empty_fields)

    explanation = explanation_block[start.end():].strip()
    if not explanation or EXTRA_FIELD.search(explanation):
        return dict(status="malformed", **empty_fields)

    letter = label[1].upper()
    return dict(status="ok", explanation=explanation, label_letter=letter,
                predicted_label=order[LETTERS.index(letter)])


def make_jobs(examples, edits):
    edits_by_source = {}
    for edit in edits:
        if edit["status"] == "proposed":
            edits_by_source.setdefault(edit["source_id"], []).append(edit)

    jobs = []
    for example in examples:
        source_id = example["source_id"]
        variants = [("baseline", "", example["premise"], example["hypothesis"])]
        variants.extend((edit["family"], edit["edit_id"], edit["edited_premise"], edit["edited_hypothesis"])
                        for edit in edits_by_source.get(source_id, []))
        for layout, order in enumerate(OPTION_ORDERS, start=1):
            for family, edit_id, premise, hypothesis in variants:
                jobs.append(dict(
                    job_id=f"{source_id}|{layout}|{family}", source_id=source_id,
                    layout=layout, family=family, edit_id=edit_id,
                    original_gold_label=example["gold_label"],
                    option_a=order[0], option_b=order[1], option_c=order[2],
                    premise=premise, hypothesis=hypothesis, prompt=make_prompt(premise, hypothesis, order),
                ))
    return jobs

smoke_jobs = make_jobs(smoke_examples, smoke_edits)
print(f"Validation pairs: {len(smoke_examples)}. Planned generations: {len(smoke_jobs)}.")

### Load Qwen and generate a batch

Load the pinned [Qwen3-8B revision](https://huggingface.co/Qwen/Qwen3-8B/tree/c4991a667c9137cff07024144d448d2afe76abc4) in FP16 without quantization. Split the weights across the available GPUs, use the model's chat template with thinking disabled, and decode greedily. Each prompt gets its own chat context.

In [ ]:
from time import perf_counter

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

gpu_count = torch.cuda.device_count()
gpu_memory = [torch.cuda.get_device_properties(i).total_memory / 2**30 for i in range(gpu_count)]
if not gpu_memory or (gpu_count == 1 and gpu_memory[0] < 18):
    raise RuntimeError("Select GPU T4 x2 in Kaggle, or use a GPU with at least 18 GiB of memory.")

load_start = perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float16, device_map="balanced",
).eval()
if any(str(device) in {"cpu", "disk"} for device in model.hf_device_map.values()):
    raise RuntimeError("Qwen did not fit fully on the GPUs. Check the selected accelerator.")

model_load_seconds = perf_counter() - load_start
eos = model.generation_config.eos_token_id
if eos is None:
    eos = tokenizer.eos_token_id
eos_tokens = {eos} if isinstance(eos, int) else set(eos)
generation = GenerationConfig(
    do_sample=False, max_new_tokens=MAX_NEW_TOKENS, use_cache=True,
    eos_token_id=eos, pad_token_id=tokenizer.pad_token_id,
)
input_device = model.get_input_embeddings().weight.device

run_settings = {
    "model": MODEL_ID, "revision": MODEL_REVISION, "dtype": "float16", "quantized": False,
    "enable_thinking": False, "do_sample": generation.do_sample,
    "use_model_defaults": False, "batch_size": BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS, "cuda": torch.version.cuda, "python": sys.version.split()[0],
    "device_map": {name: str(device) for name, device in model.hf_device_map.items()},
    "gpus": [torch.cuda.get_device_name(i) for i in range(gpu_count)], "gpu_memory_gib": gpu_memory,
    "versions": {name: version(name) for name in ("torch", "transformers", "accelerate")},
}
print(f"Loaded {MODEL_ID} in {model_load_seconds:.1f}s on {', '.join(run_settings['gpus'])}.")

@torch.inference_mode()
def generate_batch(jobs):
    chats = [[{"role": "user", "content": job["prompt"]}] for job in jobs]
    inputs = tokenizer.apply_chat_template(
        chats, tokenize=True, add_generation_prompt=True, enable_thinking=False,
        padding=True, return_dict=True, return_tensors="pt",
    ).to(input_device)
    prompt_width = inputs["input_ids"].shape[1]
    generated = model.generate(
        **inputs, generation_config=generation, use_model_defaults=run_settings["use_model_defaults"],
    )[:, prompt_width:].tolist()

    results = []
    for job, tokens in zip(jobs, generated, strict=True):
        stop = next((i for i, token in enumerate(tokens) if token in eos_tokens), None)
        text = tokenizer.decode(tokens[:stop], skip_special_tokens=False,
                                clean_up_tokenization_spaces=False)
        results.append({**job, "raw_response": text, "generated_tokens": len(tokens[:stop]),
                        "finish_reason": "eos" if stop is not None else "length"})
    return results

### Save each batch so the run can resume

After each batch, save the raw responses to SQLite. When the notebook reopens the checkpoint, it checks that the prompts and settings match and skips the saved batches. The smoke test checks this by saving eight responses, reopening the checkpoint, and finishing the run.

In [ ]:
import sqlite3
from contextlib import closing

def prepare_checkpoint(db, signature):
    with db:
        db.execute("CREATE TABLE IF NOT EXISTS run (signature TEXT PRIMARY KEY)")
        db.execute("CREATE TABLE IF NOT EXISTS responses (job_id TEXT PRIMARY KEY, payload TEXT)")
        saved_signature = db.execute("SELECT signature FROM run").fetchone()
        if saved_signature is None:
            db.execute("INSERT INTO run VALUES (?)", (signature,))
        elif saved_signature[0] != signature:
            raise ValueError("Checkpoint belongs to a different job list or run setup.")


def run_jobs(jobs, checkpoint, settings, generate, batch_size, max_batches=None):
    signature = sha256(json.dumps([settings, jobs], sort_keys=True).encode("utf-8")).hexdigest()
    checkpoint.parent.mkdir(parents=True, exist_ok=True)

    with closing(sqlite3.connect(checkpoint)) as db:
        prepare_checkpoint(db, signature)

        records = {job_id: json.loads(payload)
                   for job_id, payload in db.execute("SELECT job_id, payload FROM responses")}
        reused = len(records)
        batches_run = 0
        for batch_index, start in enumerate(range(0, len(jobs), batch_size)):
            batch = jobs[start:start + batch_size]
            if all(job["job_id"] in records for job in batch):
                continue
            if max_batches is not None and batches_run >= max_batches:
                break

            started = perf_counter()
            generated = generate(batch)
            elapsed = perf_counter() - started
            rows = [{**result, "batch_index": batch_index, "batch_seconds": elapsed}
                    for result in generated]
            if [row["job_id"] for row in rows] != [job["job_id"] for job in batch]:
                raise ValueError("Generation results do not match the planned jobs.")
            with db:
                db.executemany("INSERT INTO responses VALUES (?, ?)",
                               [(row["job_id"], json.dumps(row, ensure_ascii=False)) for row in rows])
            records.update((row["job_id"], row) for row in rows)
            batches_run += 1
            print(f"Saved {len(records)}/{len(jobs)} responses ({elapsed:.1f}s for this batch).")

    return [records[job["job_id"]] for job in jobs if job["job_id"] in records], reused

### Run the check and export the results

The summary reports parsing failures, the number of responses reused after reopening the checkpoint, GPU memory, and a rough projection for the test split. The projection uses generation time and includes all proposed test edits before audit exclusions.

In [ ]:
checkpoint = SMOKE_OUTPUT / "checkpoint.sqlite3"
first_results, _ = run_jobs(smoke_jobs, checkpoint, run_settings, generate_batch, BATCH_SIZE, max_batches=1)
smoke_results, reused = run_jobs(smoke_jobs, checkpoint, run_settings, generate_batch, BATCH_SIZE)
assert reused == len(first_results) and smoke_results[:reused] == first_results
assert len(smoke_results) == len(smoke_jobs)

for row in smoke_results:
    order = [row[f"option_{letter.lower()}"] for letter in LETTERS]
    row.update(parse_response(row["raw_response"], row["finish_reason"] == "eos", order))

export_csv(SMOKE_OUTPUT / "responses.csv", smoke_results)
batch_times = {row["batch_index"]: row["batch_seconds"] for row in smoke_results}
generation_seconds = sum(batch_times.values())
status_counts = Counter(row["status"] for row in smoke_results)
baselines = [row for row in smoke_results if row["family"] == "baseline" and row["status"] == "ok"]
baseline_correct = sum(row["predicted_label"] == row["original_gold_label"] for row in baselines)
full_job_estimate = len(OPTION_ORDERS) * (
    len(test_examples) + sum(row["status"] == "proposed" for row in test_edits)
)
smoke_summary = {
    **run_settings, "validation_examples": len(smoke_examples), "generations": len(smoke_results),
    "status_counts": dict(status_counts), "resume_reused": reused,
    "parsed_baselines": len(baselines), "correct_baselines": baseline_correct,
    "generation_seconds": generation_seconds, "model_load_seconds": model_load_seconds,
    "estimated_full_generations_before_audit": full_job_estimate,
    "estimated_full_generation_hours": generation_seconds * full_job_estimate / len(smoke_results) / 3600,
    "peak_reserved_gib": [round(torch.cuda.max_memory_reserved(i) / 2**30, 2) for i in range(gpu_count)],
}
(SMOKE_OUTPUT / "summary.json").write_text(json.dumps(smoke_summary, indent=2) + "\n", encoding="utf-8")
print(f"Response statuses: {dict(status_counts)}. Reused after reopening: {reused}.")
print(f"Baseline accuracy among parsed answers: {baseline_correct}/{len(baselines)}.")
print(f"Projected generation time before audit exclusions: {smoke_summary['estimated_full_generation_hours']:.2f}h.")
print(f"Outputs: {SMOKE_OUTPUT.resolve()}.")